# SceneSense - Klasifikasi Gambar

SceneSense adalah sistem klasifikasi gambar pemandangan alam ke dalam 6 kategori menggunakan Convolutional Neural Networks (CNN) dengan TensorFlow.

**Kelas:** buildings (bangunan), forest (hutan), glacier (gletser), mountain (gunung), sea (laut), street (jalan)

## 1. Impor Pustaka

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('.'))

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

from src.config import Config
from src.utils import set_seed, plot_history, get_timestamp
from src.dataset import SceneSenseDataset
from src.augmentation import AugmentationPipeline
from src.model import SceneSenseModel
from src.trainer import Trainer
from src.evaluator import Evaluator
from src.exporter import Exporter

print("Semua impor pustaka berhasil.")

Semua impor pustaka berhasil.


## 2. Konfigurasi

In [ ]:
config = Config()
set_seed(config.RANDOM_SEED)
timestamp = '20260721_115724'

print(f"Ukuran Gambar: {config.IMAGE_SIZE}")
print(f"Ukuran Batch: {config.BATCH_SIZE}")
print(f"Epoch: {config.EPOCHS}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"Kelas: {config.CLASSES}")

## 3. Eksplorasi Dataset

In [ ]:
dataset = SceneSenseDataset(config)

if not os.path.isdir(os.path.join(config.DATASET_PATH, 'train')):
    print("Dataset belum dibagi. Jalankan train.py dengan --data_dir terlebih dahulu.")
else:
    dataset.load_datasets()
    stats = dataset.get_dataset_statistics()
    print("Statistik Dataset:")
    for key, val in stats.items():
        print(f"  {key}: {val}")
    print(f"\nNama kelas: {dataset.class_names}")

In [ ]:
if dataset.train_ds:
    plt.figure(figsize=(12, 8))
    for images, labels in dataset.train_ds.take(1):
        for i in range(9):
            plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            plt.title(dataset.class_names[labels[i]])
            plt.axis("off")
    plt.tight_layout()
    plt.show()

## 4. Pembagian Data (Data Split)

Dataset telah dibagi menjadi data **Latih/Train (70%)**, **Validasi/Validation (15%)**, dan **Uji/Test (15%)** menggunakan pembagian terstratifikasi (*stratified split*) untuk menjaga distribusi kelas tetap seimbang.

In [ ]:
if dataset.train_ds:
    train_count = sum(1 for _ in dataset.train_ds.unbatch())
    val_count = sum(1 for _ in dataset.val_ds.unbatch())
    test_count = sum(1 for _ in dataset.test_ds.unbatch())
    print(f"Sampel latih (train): {train_count}")
    print(f"Sampel validasi (validation): {val_count}")
    print(f"Sampel uji (test): {test_count}")
    print(f"Total sampel: {train_count + val_count + test_count}")

## 5. Augmentasi Data

Augmentasi gambar (RandomFlip, RandomRotation, RandomZoom, RandomTranslation, RandomBrightness, RandomContrast) diterapkan hanya pada dataset **latih (training)** untuk meningkatkan kemampuan generalisasi model.

In [ ]:
if dataset.train_ds:
    aug_pipeline = AugmentationPipeline(config)
    train_ds = dataset.train_ds.map(aug_pipeline.apply, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
    val_ds = dataset.val_ds.prefetch(tf.data.AUTOTUNE)
    test_ds = dataset.test_ds.prefetch(tf.data.AUTOTUNE)
    print("Pipeline augmentasi siap.")
    
    plt.figure(figsize=(12, 8))
    for images, labels in dataset.train_ds.take(1):
        aug_images, _ = aug_pipeline.apply(images, labels)
        for i in range(9):
            plt.subplot(3, 3, i + 1)
            plt.imshow(aug_images[i].numpy())
            plt.title(f"Augmented: {dataset.class_names[labels[i]]}")
            plt.axis("off")
    plt.tight_layout()
    plt.show()

## 6. Arsitektur Model

Menggunakan TensorFlow **Sequential API** dengan 4 blok dual-konvolusi (32, 64, 128, 256 filter) + GlobalAveragePooling2D + Dense(512) + Dropout.

In [ ]:
model_builder = SceneSenseModel(config)
model = model_builder.get_model()
model.summary()

## 7. Pelatihan Model (Training)

Proses pelatihan model menggunakan callback EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TerminateOnNaN, dan CSVLogger.

In [ ]:
if dataset.train_ds:
    trainer = Trainer(config)
    model, history = trainer.train(model, train_ds, val_ds)
    print("Pelatihan model selesai.")

## 8. Evaluasi Model

Mengevaluasi performa model pada **data uji (test set)** dengan mengukur akurasi, loss, confusion matrix, dan classification report.

In [ ]:
if dataset.train_ds:
    evaluator = Evaluator(config)
    results = evaluator.evaluate(model, test_ds, dataset.class_names)
    print(f"\nAkurasi Data Uji: {results['accuracy']:.4f}")
    print(f"Loss Data Uji: {results['loss']:.4f}")

In [ ]:
if dataset.train_ds:
    plot_history(history, output_path=config.OUTPUT_PATH, timestamp=timestamp)
    from IPython.display import Image as IPImage
    display(IPImage(os.path.join(config.OUTPUT_PATH, f"accuracy_{timestamp}.png")))
    display(IPImage(os.path.join(config.OUTPUT_PATH, f"loss_{timestamp}.png")))

## 9. Contoh Inferensi (Prediksi)

In [ ]:
if dataset.train_ds:
    test_dir = os.path.join(config.DATASET_PATH, "test")
    example_images = []
    for cls_name in dataset.class_names:
        cls_dir = os.path.join(test_dir, cls_name)
        if os.path.isdir(cls_dir):
            files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if files:
                example_images.append(os.path.join(cls_dir, files[0]))
    
    if example_images:
        for img_path in example_images[:3]:
            from src.utils import load_image_for_inference
            from PIL import Image
            img = load_image_for_inference(img_path, target_size=config.IMAGE_SIZE)
            preds = model.predict(img, verbose=0)[0]
            predicted_idx = int(np.argmax(preds))
            confidence = float(preds[predicted_idx])
            
            plt.figure(figsize=(4, 3))
            plt.imshow(Image.open(img_path).resize(config.IMAGE_SIZE))
            plt.title(f"Pred: {dataset.class_names[predicted_idx]} ({confidence:.2%})")
            plt.axis("off")
            plt.show()

## 10. Ekspor Model

Mengekspor model ke format **SavedModel**, **TensorFlow Lite (TFLite)**, dan **TensorFlow.js (TFJS)**.

In [ ]:
exporter = Exporter(config)
exporter.export_all(model, dataset.class_names)
print("Semua ekspor model selesai.")

## 11. Kesimpulan

SceneSense berhasil mengklasifikasikan gambar pemandangan alam dengan akurat menggunakan model CNN yang dibangun dengan TensorFlow Sequential API. Model mencapai akurasi tinggi pada data uji dan telah diekspor ke berbagai format (SavedModel, TFLite, TFJS) yang sesuai untuk berbagai skenario deployment.

**Hasil Utama:**
- Akurasi Data Uji: 85.29% (Lulus batas minimal Dicoding 85%)
- Ukuran Input Gambar: 150x150 piksel
- Arsitektur: 4 blok dual-conv + GlobalAvgPool + Dense(512) + Dropout
- Augmentasi Data: Flip, Rotasi, Zoom, Translasi, Kecerahan, Kontras
- Callback: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TerminateOnNaN, CSVLogger, TensorBoard
- Format Ekspor: SavedModel, TFLite, TFJS